In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import astropy.units as u
from astropy.units import cds
from tabulate import tabulate
from scipy.interpolate import interp1d

root = Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from stepsic.parameters import CosmoParameters
from stepsic.data import CosmoData

import logging
log = logging.getLogger(__name__)
logging.basicConfig(level=logging.WARNING)

### Physical parameters

$$
    \omega_b = \frac{\omega_b h^{2}}{h^{2}} \qquad\text{with}\quad h=\tfrac{H_0}{100}.
$$

with propagated errors as

$$
    \sigma_{\omega_b}^2
    \;=\;
    \left[ \left(\frac{\sigma_{\omega_b h^2}}{\omega_b h^2}\right)^{\!2}
    \;+\;
    \left(2\,\frac{\sigma_{H_0}}{H_0}\right)^{\!2}\; \right] \rule{0pt}{14pt}\omega_b^{\;2}\,.
$$

In [ ]:
def calculate_omx(omxh2, omxh2_err, H0, H0_err, name=None):
    h = H0 / 100
    omx = omxh2 / h**2
    omx_err = np.sqrt(((omxh2_err / omxh2)**2 + (2*H0_err / H0)**2) * omx**2)
    print(f'{name}:\t{omx:.8f} +- {omx_err:.8f}')
    return omx, omx_err


In [ ]:
# 2.17
omega_b, omega_b_err = calculate_omx(
        omxh2=0.022383, omxh2_err=0.0, H0=67.32, H0_err=0.0, name='best fit 2.17')
omega_b, omega_b_err = calculate_omx(
        omxh2=0.02237, omxh2_err=0.00015, H0=67.36, H0_err=0.54, name=r'68% limit 2.17')
# 2.18
omega_b, omega_b_err = calculate_omx(
        omxh2=0.022447, omxh2_err=0.0, H0=67.702, H0_err=0.0, name='best fit 2.18')
omega_b, omega_b_err = calculate_omx(
        omxh2=0.02242, omxh2_err=0.00014, H0=67.66, H0_err=0.42, name=r'68% limit 2.18')
# 2.19
omega_b, omega_b_err = calculate_omx(
        omxh2=0.022408, omxh2_err=0.0, H0=67.49, H0_err=0.0, name='best fit 2.19')
omega_b, omega_b_err = calculate_omx(
        omxh2=0.02239, omxh2_err=0.00014, H0=67.48, H0_err=0.50, name=r'68% limit 2.19')
# 2.20
omega_b, omega_b_err = calculate_omx(
        omxh2=0.022436, omxh2_err=0.0, H0=67.742, H0_err=0.0, name='best fit 2.20')
omega_b, omega_b_err = calculate_omx(
        omxh2=0.02243, omxh2_err=0.00013, H0=67.72, H0_err=0.40, name=r'68% limit 2.20')

In [ ]:
# Gravitational constant in cm^3 g^-1 s^-2
G = 1 * cds.G

# This IC generator uses the same internal units as the StePS simulator code
UNIT_L = (1 * u.Mpc).to(u.cm)              # Unit distance (1 Mpc in cm)
UNIT_M = (1e11 * u.M_sun).to(u.g)          # Unit mass (1e11 Msol in g)
UNIT_T = np.sqrt(UNIT_L**3 / G / UNIT_M).to(u.Gyr)  # Unit time
UNIT_V = UNIT_L.to(u.km) / UNIT_T.to(u.s)  # Unit velocity

### Resolution-Mass Map

In [ ]:
params = CosmoParameters(path=Path('config.toml')).get_parameters()

In [ ]:
ic = CosmoData.load_snapshot(Path(params['INPUT_GLASS']))
ic.to_internal_units(params)
ic.rescale_snapshot_mass(params)
ic.periodic_shift(params)

#### Original

In [ ]:
res = np.cbrt(ic.M_box/ic.mass_list).astype(int)
res_mass_map = np.c_[ic.mass_list, res]
res_tab = np.zeros(params['NGRIDSAMPLES'], dtype=int)
mass_tab = np.zeros(params['NGRIDSAMPLES'], dtype=int)

delta_Nsample = np.ceil(ic.mass_list.size/params['NGRIDSAMPLES']).astype(int)

print("The generated Nsample list:")
print("ID\tNsample\tMass(in 10e11Msol)")
res_tab[-1] = res_mass_map[0, 1]
print("%i\t%i\t%d" % (len(res_tab)-1, res_tab[-1], res_mass_map[0, 0]))
for i in range(len(res_tab)-2, -1, -1):
    mass_tab[i] = res_mass_map[-1-i*delta_Nsample, 0]
    res_tab[i] = res_mass_map[-1-i*delta_Nsample, 1]
    print("%i\t%i\t%d" % (i, res_tab[i], mass_tab[i]))

#### Reworked

In [ ]:
def create_nres_mass_map(n_grid_samples, mass_list, M_box, Lbox):
    '''
    Creates a lookup table for the number of voxels per mass bin
    for a variable resolution grid in a regular StePS simulation.

    Parameters
    ----------
    n_grid_samples : int
        Number of grids with different resolutions.
    mass_list : ndarray
        Array containing the sorted unique particle masses.
    M_box : float
        Total mass in the simulation box (in 1e11 Msol).
    Lbox : ndarray
        Box dimensions in [Mpc]. Can be a single scalar for a cubical box or
        an array in the form of `[Lx, Ly, Lz]` for a rectangular cuboid.

    Returns
    -------
    nres_tab : ndarray of shape (n_grid_samples,)
        Array containing the number of resolution elements for each grid.
    mass_tab : ndarray of shape (n_grid_samples,)
        Array containing the mass values corresponding to each grid.
    '''
    nres_list = np.min(Lbox) // np.cbrt(np.prod(Lbox) * mass_list / M_box)
    idx = np.linspace(
        0, mass_list.size - 1, n_grid_samples, endpoint=True, dtype=int)
    
    # Populate lookup table by starting with the outermost mass bin
    nres_tab = nres_list[idx[::-1]]
    mass_tab = mass_list[idx[::-1]]

    log.info('The generated resolution-mass map:')
    print(tabulate([*zip(nres_tab, mass_tab)],
                   headers=['Resolution', 'Mass [1e11Msol]'], floatfmt='.0f'))
    return nres_tab, mass_tab

In [ ]:
nres_tab, mass_tab = create_nres_mass_map(
    params['NGRIDSAMPLES'], ic.mass_list, ic.M_box, params['LBOX'])

In [ ]:
def test_interp():
    disp = np.random.random(params['NGRIDSAMPLES'])
    disp_interp = np.interp(ic.mass, mass_tab, disp)
    return any(disp_interp == disp[-1])
print('Interpolation always returns the highest res displacement:')
any([test_interp() for _ in range(1000)])

### Interpolation

In [ ]:
disp = np.random.random(size=(params['NGRIDSAMPLES'], ic.mass.size, 3))
for i in range(0, ic.mass.size):
    for k in range(0, 3):
        y = np.interp(ic.mass[i], mass_tab, disp[:, i, k])

### Generate file names

In [ ]:
def create_fname(params):
    '''
    Construct a filename for the output IC based on the parameters.

    Parameters
    ----------
    params : dict
        Dictionary containing the simulation parameters.

    Returns
    -------
    str
        The generated filename.
    '''
    fname = f"{params['IC_PREFIX']}_"
    fname += "Lx{}_Ly{}_Lz{}_".format(*map(int, params['LBOX']))
    fname += f"D4D{params['D_4D']:.0f}_z{params['REDSHIFT']:.0f}"
    return fname

In [ ]:
create_fname(params)

### Regex match and collect input files

In [ ]:
def detect_extension(path: Path):
    '''
    Detects the file extension of the given path.

    Parameters
    ----------
    path : pathlib.Path
        The path to the file whose extension is to be detected.

    Returns
    -------
    match : re.Match or None
        A match object if the path matches a valid pattern, or `None` if
        it does not.
    '''
    parser_re = re.compile(
        r"^(?P<stem>.*?)"
        r"(?:"
        # Pattern 1: .<index#.<ext# (e.g. ".0.hdf5")
        r"\.(?P<index1>\d+)\.(?P<ext1>[a-zA-Z_][a-zA-Z0-9_]*)"
        r"|"
        # Pattern 2: .<ext>.<index> (e.g. ".hdf5.0")
        r"\.(?P<ext2>[a-zA-Z_][a-zA-Z0-9_]*)\.(?P<index2>\d+)"
        r"|"
        # Pattern 3: .<ext> (e.g. ".hdf5")
        r"\.(?P<ext3>[a-zA-Z_][a-zA-Z0-9_]*)"
        r")$"
    )
    return parser_re.match(path.name)

In [ ]:
def collect_files(path: Path):
    '''
    Gathers all snapshot files belonging to the same group.

    Parameters
    ----------
    path : pathlib.Path
        The path to any single file in the snapshot set.

    Returns
    -------
    files : List[pathlib.Path]
        A sorted list of Path objects for all files in the snapshot.
        Returns an empty list if the filepath does not match a valid
        pattern.
    '''
    match = detect_extension(path)

    if match:
        # Regex looking for: \.digits\.ext OR \.ext\.digits OR \.ext
        parts = match.groupdict()
        stem = re.escape(parts['stem'])
        ext = re.escape(parts['ext1'] or parts['ext2'] or parts['ext3'])
        search_pattern = rf'^{stem}(\.\d+\.{ext}|\.{ext}\.\d+|\.{ext})$'
    else:
        # If no extension is provided, treat it as a gadget file
        ext_pattern = re.compile(r"^(?P<stem>.*?)(?:\.(?P<index>\d+))?$")
        match = ext_pattern.match(path.name)
        if not match:
            return [path.name]  # Standalone gadget snapshot
        search_pattern = rf'^{re.escape(match.group("stem"))}(?:\.\d+)?$'

    search_re = re.compile(search_pattern)

    files = sorted([
        f for f in path.parent.iterdir()
        if f.is_file() and search_re.match(f.name)
    ])
    
    return files

In [ ]:
files_to_create = [
    'sim_run.0.hdf5', 'sim_run.1.hdf5', 'sim_run_analysis.hdf5',
    'sim_run.0.dat', 'sim_run.1.dat', 'sim_run.2.dat', # Same stem, different ext
    'other.dat.0', 'other.dat.1', 'gadget.0', 'gadget.1', 'gadget.2',
    'gadget_alone', 'gadget_alone.log', 'gadget.log'
]
Path('./test').mkdir(parents=True, exist_ok=True)
for f in files_to_create:
    (Path('./test') / f).touch()

In [ ]:
# Case 1: Find all '.hdf5' files for 'sim_run' without explicit index
path1 = Path('./test/sim_run.hdf5')
snapshot_files1 = collect_files(path1)
print(f"Files for '{path1.name}': {[f.name for f in snapshot_files1]}")
# Expected: ['sim_run.0.hdf5', 'sim_run.1.hdf5']

# Case 2: Find all '.dat' files for 'sim_run' with explicit random index
path2 = Path('./test/sim_run.1.dat')
snapshot_files2 = collect_files(path2)
print(f"Files for '{path2.name}': {[f.name for f in snapshot_files2]}")
# Expected: ['sim_run.0.dat', 'sim_run.1.dat', 'sim_run.2.dat']

# Case 3: Find files with the alternate '.ext.index' naming
path3 = Path('./test/other.dat')
snapshot_files3 = collect_files(path3)
print(f"Files for '{path3.name}': {[f.name for f in snapshot_files3]}")
# Expected: ['other.dat.0', 'other.dat.1']

# Case 4: Standalone "Gadget" file
path4 = Path('./test/gadget_alone')
snapshot_files4 = collect_files(path4)
print(f"Files for '{path4.name}': {[f.name for f in snapshot_files4]}")
# Expected: ['gadget_alone']

# Case 5: All Gadget files in the same group
path5 = Path('./test/gadget')
snapshot_files5 = collect_files(path5)
print(f"Files for '{path5.name}': {[f.name for f in snapshot_files5]}")
# Expected: ['gadget.0', 'gadget.1', 'gadget.2']

# Case 6: A file that should NOT be grouped with the Gadget files
path6 = Path('./test/gadget.log')
snapshot_files6 = collect_files(path6)
print(f"Files for '{path6.name}': {[f.name for f in snapshot_files6]}")
# Expected: ['gadget.log']